<a href="https://colab.research.google.com/github/izzat-ai/learning-ai/blob/main/pandas/projects/MedCare_Analytics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Toshkentdagi xususiy klinika bizni yolladi. Ularning 3 yillik bemorlar ma'lumotlari bor, lekin hech qanday tahlil qilinmagan. Bizning vazifamiz — bu ma'lumotlardan amaliy biznes qarorlar chiqarish**

In [11]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

- *Dataset sun'iy intellektdan olindi*

In [12]:
df = pd.read_csv("/content/medcare_dataset.csv")
df.head()

,bemor_id,ism,yosh,jinsi,viloyat,bo_lim,shifokor,tashxis,qabul_sanasi,yotish_kunlari,narx_sum,tolov_turi,holat,qayta_murojaat,reyting
0,P00001,Bemor_1,52,Ayol,Farg'ona,Nevrologiya,Dr. Rahimov S,Migren,2022-02-20,14.0,2999095.0,Karta,Og'ir,True,1.0
1,P00002,Bemor_2,15,Erkak,Surxondaryo,Kardiologiya,Dr. Toshmatov B,Yurak yetishmovchiligi,2024-04-02,9.0,1465502.0,Naqd,Surunkali,True,3.0
2,P00003,Bemor_3,72,Erkak,Surxondaryo,Pediatriya,Dr. Tursunova G,Ich ketish,2023-03-28,23.0,4530088.0,Naqd,Surunkali,False,3.0
3,P00004,Bemor_4,61,Ayol,Qashqadaryo,Jarrohlik,Dr. Abdullayev K,Churraq,2024-07-02,6.0,1349110.0,Sug'urta,Yaxshilandi,True,4.0
4,P00005,Bemor_5,21,Erkak,Farg'ona,Jarrohlik,Dr. Abdullayev K,O'tkir appenditsit,2024-01-02,7.0,996462.0,Naqd,Yaxshilandi,False,3.0


In [13]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2015 entries, 0 to 2014
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   bemor_id        2015 non-null   object 
 1   ism             2015 non-null   object 
 2   yosh            2015 non-null   int64  
 3   jinsi           2015 non-null   object 
 4   viloyat         1990 non-null   object 
 5   bo_lim          2015 non-null   object 
 6   shifokor        2015 non-null   object 
 7   tashxis         2015 non-null   object 
 8   qabul_sanasi    2015 non-null   object 
 9   yotish_kunlari  1955 non-null   float64
 10  narx_sum        1985 non-null   float64
 11  tolov_turi      2015 non-null   object 
 12  holat           2015 non-null   object 
 13  qayta_murojaat  2015 non-null   bool   
 14  reyting         1975 non-null   float64
dtypes: bool(1), float64(3), int64(1), object(10)
memory usage: 222.5+ KB


In [14]:
df.shape

(2015, 15)

In [15]:
df.describe()

,yosh,yotish_kunlari,narx_sum,reyting
count,2015.000000,1955.000000,1.985000e+03,1975.000000
mean,41.184615,14.325831,2.533063e+06,3.852152
std,23.978902,8.649002,1.392165e+06,1.159697
min,1.000000,0.000000,1.500370e+05,1.000000
25%,20.000000,7.000000,1.340064e+06,3.000000
50%,40.000000,14.000000,2.493145e+06,4.000000
75%,62.000000,22.000000,3.731877e+06,5.000000
max,84.000000,29.000000,4.999666e+06,5.000000


- be'morlarning o'rtachasi 41 yosh ekan . Eng keksasi 84 yoshgacha bo'lsa , eng kichigi 1 yosh .
- be'morlar o'rtacha 14 kun davomida qolib davolanishgan . Ularning aksari (75% izi) 22 kun davomida yotishgan . Qolgan 25% esa bir hafta yotib davolanishgan .
- klinikada maksimal ravishda ko'p qolganlar - 5 milliongacha , o'rtacha esa 2.5 milliongacha to'lov qilishgan
- berilgan reytinglardan - be'morlarning klinikaga nisbatan bergan ballarini ko'rish mumkin . Be'morlarning 50% izi 4 reytingni bergan , bundan - ularning kamida yarmi klinikaga 4 va 5 baho bergan , bu yaxshi natija .

In [16]:
df.isnull().sum()

,0
bemor_id,0
ism,0
yosh,0
jinsi,0
viloyat,25
bo_lim,0
shifokor,0
tashxis,0
qabul_sanasi,0
yotish_kunlari,60


- viloyat, yotish_kunlari, narx_sum va reyting ustunlarida tushib qolgan qiymatlar bor , ularni oldini olish kerak

In [19]:
df.duplicated().sum()

np.int64(15)

- datasetimizda bir xil ma'lumotlardan bir nechta borligi aniq bo'ldi

In [21]:
df['tashxis'].value_counts()

,count
tashxis,
Bronxit,110
Angina,87
Qalqonsimon bez,81
Travma,77
Osteoxondroz,76
Neyropatiya,74
Mioma,70
Ovarian kista,68
Semirishlik,67


- bemorlarda turli kasalliklar qayd etilgan . Buni tashhis natijalaridan ham bilib olsak bo'ladi